#### Error fixing LLM 

In [43]:
import os,sys
sys.path.insert(0,'../libs')
sys.path.insert(0,'../src')
from utils import load_json
import openai
from llm_utils import BSAgent
import numpy as np
import pandas as pd
import re,json,copy
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from huggingface_hub import snapshot_download
from prompts import topic_identify_simple_pt

/root/miniconda3/envs/gpt/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import Optional,List,Literal,Iterable
from langchain.output_parsers import PydanticOutputParser
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI
import openai

In [44]:
# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "token-abc123"
openai_api_base = "http://localhost:8000/v1"

client = openai.OpenAI(api_key=openai_api_key,
                             base_url=openai_api_base)
model_name = client.models.list().data[0].id
error_fixing_llm = ChatOpenAI(openai_api_base=openai_api_base,
                              openai_api_key=openai_api_key,
                              model = model_name)


llm_agent  = BSAgent(api_key=openai_api_key,
                     api_base_url=openai_api_base,
                    temperature=0)
llm_agent.model = llm_agent.client.models.list().data[0].id
print('current running model :{}'.format(llm_agent.model))

current running model :/root/data/hf_cache/meta-llama/Meta-Llama-3.1-8B-Instruct


In [4]:
messages = [
    (
        "system",
        "You are a helpful assistant",
    ),
    ("human", "I love programming."),
]
ai_msg = error_fixing_llm.invoke(messages)

In [5]:
ai_msg.content

"Programming can be a fascinating and rewarding field. What kind of programming are you interested in? Are you working on a specific project or exploring different languages?\n\nSome popular programming languages include:\n\n1. Python: Known for its simplicity and versatility, Python is widely used in data science, machine learning, and web development.\n2. Java: A popular language for Android app development, Java is also used in enterprise software development and web development.\n3. JavaScript: Used for client-side scripting on the web, JavaScript is also popular for building desktop and mobile applications.\n4. C++: A high-performance language, C++ is commonly used for building operating systems, games, and other high-performance applications.\n5. Swift: Developed by Apple, Swift is a modern language for building iOS, macOS, watchOS, and tvOS apps.\n\nIf you're just starting out, I'd be happy to help you choose a language or provide resources for learning."

In [6]:
class Topic(BaseModel):
    topic_label: Literal["Economic Outlook", "Monetary Policy", "Fiscal Stance", 
                         "Financial Stability","External Stance","Climate Change",
                         "Inclusion","Technology","Governance","Structural Reforms",
                         "Other Topics"] = Field(description="short title for the topic")
    confidence_score: int = Field(description="confidence score of topic identification, ranges from 0 to 100")

class Topic_Result(BaseModel):
    reasoning: str = Field(description="the reasoning process for topic identification")
    topic_labels: List[Topic] = Field(description="list of topics that are identified")
    
parser = PydanticOutputParser(pydantic_object=Topic_Result)

#pprint.pprint(parser.get_format_instructions())
example_output = '{"reasoning": "test test test", "topic_labels": [{"topic_label":"Other Topics","confidence_score":10}]}'
parser.parse(example_output)

Topic_Result(reasoning='test test test', topic_labels=[Topic(topic_label='Other Topics', confidence_score=10)])

In [7]:
from langchain.output_parsers import OutputFixingParser

In [8]:
#### it can only handel errors to some extend 
new_parser = OutputFixingParser.from_llm(parser=parser, llm=error_fixing_llm)
misformated_output = '{"reasoning": "test test test", "topic_labels": [{"topic_label":"Other Topics","confidence_scor":10}'

new_parser.parse(misformated_output)

Topic_Result(reasoning='test test test', topic_labels=[Topic(topic_label='Other Topics', confidence_score=10)])

#### Customized LLM output fixing llm 
- looks like it can fix the problem 

In [55]:

output_fixing_pt = {'System':"You are an intelligent assistant specialized in formatting text. Your task is to take raw LLM output and format it according to user-provided instructions. Follow the specific formatting guidelines given and ensure the final output is clean, readable, and adheres to the specified style.",
      'Human':"""
Here is the raw LLM output:
----------------
----------------
{LLM_OUTPUT}
----------------
----------------

You are supposed to extract appropriate information for reasoning and topic_labels. topic_labels has additional information for confidence score. 
Please respond in clean json format as follow:
```json
{{"reasoning": "<reasoning process>", 
"topic_labels": [{{"topic_label":"<identified topic label>","confidence_score":<confidence score>}},...]}}
```
Response:
"""}

llm_output = """soemrandome information before the results; alds;kfjadsjq[weirla;knxc./andf]
them here is to output : 

"reasoning": "reason for decide it as other topics", "topic_labels": 
{"topic_label":"Other Topics",
"confidence_score":20}

Then here are all the results
'
"""
output_fixing_pt['Human'] = output_fixing_pt['Human'].format(LLM_OUTPUT=llm_output)


In [57]:
res = llm_agent.get_response_content(prompt_template=output_fixing_pt, 
                                     #max_tokens=4096,
                                     temperature=0)

In [58]:
print(res)

```json
{
  "reasoning": "reason for decide it as other topics",
  "topic_labels": [
    {
      "topic_label": "Other Topics",
      "confidence_score": 20
    }
  ]
}
```

I have extracted the relevant information from the raw LLM output and formatted it into a clean JSON object according to your specifications.


In [59]:
parser.parse(res)

Topic_Result(reasoning='reason for decide it as other topics', topic_labels=[Topic(topic_label='Other Topics', confidence_score=20)])